## Proyecto — Data Stream Processor

In [1]:
from goodrich.ch06.array_stack import *
from goodrich.ch06.array_queue import *

In [2]:
class DataProcessor: 
    
    def __init__(self): 
        self.registros_pendientes = ArrayQueue() 
        self.historial_cambios = ArrayStack() 
        self.list = [] 
 
    def add(self, dato): 
        if len(dato) != 3: 
            raise TypeError 
        
        if not isinstance(dato[2], (int, float)): 
            raise TypeError 
     
        self.registros_pendientes.enqueue(dato) 
 
    def process_next(self): 
        
        if self.registros_pendientes.is_empty():
            raise Empty

        registro = self.registros_pendientes.dequeue() 
        encontro = False 
 
        for posicion, i in enumerate(self.list): 
 
            dato1 = i[0] 
            dato2 = i[1] 
            dato3 = i[2] 
 
            if dato1 == registro[0] and dato2 == registro[1]:   
 
                self.historial_cambios.push((dato1, dato2, dato3)) 
                nueva_tupla = (dato1, dato2, registro[2]) 
                self.list[posicion] = nueva_tupla 
 
                encontro = True 
 
        if encontro == False: 
 
            self.list.append(registro) 
            nueva_tupla2 = (registro[0], registro[1], None) 
            self.historial_cambios.push(nueva_tupla2) 
 
        return registro 
 
    def undo(self): 
        
        if self.historial_cambios.is_empty():
            raise Empty
        
        cambio = self.historial_cambios.pop() 
 
        dato1 = cambio[0] 
        dato2 = cambio[1] 
        valor_anterior = cambio[2] 
 
        for posicion, registro in enumerate(self.list): 
 
            if registro[0] == dato1 and registro[1] == dato2: 
 
                if valor_anterior is None: 
                    self.list.pop(posicion) 
                else: 
                    self.list[posicion] = (dato1, dato2, valor_anterior) 
 
                return 
 
    def pending(self): 
        return len(self.registros_pendientes) 
     
    def current_value(self, sensor, variable): 
        
        for i in self.list: 
 
            dato1 = i[0] 
            dato2 = i[1] 
            dato3 = i[2] 
 
            if dato1 == sensor and dato2 == variable: 
 
                return dato3 
             
        raise KeyError

## Análisis de complejidad

| Método | Complejidad temporal | Justificación |
|---|---|---|
| `add` | **O(1)** | Las validaciones y `enqueue()` son operaciones constantes. |
| `process_next` | **O(n)** | Recorre `self.list` para buscar el sensor y la variable. |
| `undo` | **O(n)** | `pop()` del Stack es O(1), pero se recorre `self.list` para encontrar el registro. |
| `pending` | **O(1)** | `len()` de la Queue es una operación constante. |
| `current_value` | **O(n)** | Recorre `self.list` hasta encontrar el sensor y la variable. |

Donde **n** es la cantidad de registros almacenados en `self.list`.

## Estructuras auxiliares

- **`ArrayQueue`** (`registros_pendientes`): almacena los registros pendientes de procesar siguiendo el orden **FIFO**.
- **`ArrayStack`** (`historial_cambios`): almacena el historial de cambios siguiendo el orden LIFO, permitiendo realizar `undo()`.
- **Lista de Python** (`self.list`): almacena el estado actual de los registros.

---

## Pruebas

In [19]:
procesador = DataProcessor()


# 1. Probar que inicialmente no hay registros pendientes
print("1. Pendientes inicialmente:", len(procesador.registros_pendientes))

1. Pendientes inicialmente: 0


In [20]:
# 2. Probar add()
registro1 = ("S01", "temperatura", 20)
procesador.add(registro1)

print("2. Pendientes después de add:", len(procesador.registros_pendientes))

2. Pendientes después de add: 1


In [21]:
# 3. Probar agregar varios registros
registro2 = ("S02", "humedad", 60)
registro3 = ("S03", "presion", 1013)

procesador.add(registro2)
procesador.add(registro3)

print("3. Pendientes después de varios add:", len(procesador.registros_pendientes))

3. Pendientes después de varios add: 3


In [22]:
# 4. Probar FIFO
print("4. Primer registro de la Queue:", procesador.registros_pendientes.first())

4. Primer registro de la Queue: ('S01', 'temperatura', 20)


In [23]:
# 5. Probar process_next()
procesado = procesador.process_next()

print("5. Registro procesado:", procesado)
print("   Lista actual:", procesador.list)

5. Registro procesado: ('S01', 'temperatura', 20)
   Lista actual: [('S01', 'temperatura', 20)]


In [24]:
# 6. Probar procesar varios registros
procesado2 = procesador.process_next()
procesado3 = procesador.process_next()

print("6. Segundo registro procesado:", procesado2)
print("   Tercer registro procesado:", procesado3)
print("   Pendientes después de procesar:", len(procesador.registros_pendientes))
print("   Lista actual:", procesador.list)

6. Segundo registro procesado: ('S02', 'humedad', 60)
   Tercer registro procesado: ('S03', 'presion', 1013)
   Pendientes después de procesar: 0
   Lista actual: [('S01', 'temperatura', 20), ('S02', 'humedad', 60), ('S03', 'presion', 1013)]


In [25]:
# 7. Probar actualizar un registro existente
registro4 = ("S01", "temperatura", 25)

procesador.add(registro4)
procesado4 = procesador.process_next()

print("7. Registro actualizado:", procesado4)
print("   Lista después de actualizar:", procesador.list)


7. Registro actualizado: ('S01', 'temperatura', 25)
   Lista después de actualizar: [('S01', 'temperatura', 25), ('S02', 'humedad', 60), ('S03', 'presion', 1013)]


In [26]:
# 8. Probar current_value()
valor = procesador.current_value("S01", "temperatura")

print("8. Valor actual de S01/temperatura:", valor)
print(procesador.list[0])

8. Valor actual de S01/temperatura: 25
('S01', 'temperatura', 25)


In [27]:
# 9. Probar undo() de una actualización
procesador.undo()

print("9. Lista después de undo():", procesador.list)
print("   Valor restaurado:", procesador.current_value("S01", "temperatura"))

9. Lista después de undo(): [('S01', 'temperatura', 20), ('S02', 'humedad', 60), ('S03', 'presion', 1013)]
   Valor restaurado: 20


In [28]:
# 10. Probar varios undo() consecutivos
procesador.undo()
procesador.undo()

print("10. Lista después de varios undo():", procesador.list)

10. Lista después de varios undo(): [('S01', 'temperatura', 20)]


In [29]:
# 11. Probar process_next() cuando la Queue está vacía
procesador = DataProcessor()

try:
    procesador.process_next()
    
except Empty:
    print("11. Empty correcto: no hay registros pendientes")

11. Empty correcto: no hay registros pendientes


In [30]:
# 12. Probar undo() cuando el historial está vacío
try:
    procesador.undo()
    
except Empty:
    print("12. Empty correcto: no hay cambios para deshacer")

12. Empty correcto: no hay cambios para deshacer


In [31]:
# 13. Probar undo() de un registro que antes no existía
procesador.add(("S04", "temperatura", 25))
procesador.process_next()

print("13. Antes de undo():", procesador.list)

procesador.undo()

print("Después de undo():", procesador.list)

try:
    procesador.current_value("S04", "temperatura")
except KeyError:
    print("KeyError correcto: S04/temperatura fue eliminado")

13. Antes de undo(): [('S04', 'temperatura', 25)]
Después de undo(): []
KeyError correcto: S04/temperatura fue eliminado


In [32]:
# 14. Probar varios cambios sobre la misma variable
procesador = DataProcessor()

procesador.add(("S05", "temperatura", 20))
procesador.process_next()

procesador.add(("S05", "temperatura", 25))
procesador.process_next()

procesador.add(("S05", "temperatura", 30))
procesador.process_next()

print("14. Valor después de tres cambios:",
      procesador.current_value("S05", "temperatura"))

procesador.undo()
print("Después de primer undo:",
      procesador.current_value("S05", "temperatura"))

procesador.undo()
print("Después de segundo undo:",
      procesador.current_value("S05", "temperatura"))

procesador.undo()

try:
    procesador.current_value("S05", "temperatura")
    
except KeyError:
    print("Después de tercer undo: registro eliminado correctamente")

14. Valor después de tres cambios: 30
Después de primer undo: 25
Después de segundo undo: 20
Después de tercer undo: registro eliminado correctamente


In [33]:
# 15. Probar registro con formato incorrecto
procesador = DataProcessor()

try:
    procesador.add(("S06", "temperatura"))

except TypeError:
    print("15. TypeError correcto: el registro no tiene 3 componentes")

15. TypeError correcto: el registro no tiene 3 componentes


In [34]:
procesador.add(("S06", "temperatura"))

TypeError: 

In [35]:
# 16. Probar valor no numérico
try:
    procesador.add(("S06", "temperatura", "20"))
except TypeError:
    print("16. TypeError correcto: el valor no es numérico")

16. TypeError correcto: el valor no es numérico


In [36]:
procesador.add(("S06", "temperatura", "20"))

TypeError: 

In [37]:
# 17. Probar búsqueda de un registro inexistente
try:
    procesador.current_value("S99", "temperatura")
except KeyError:
    print("17. KeyError correcto: S99/temperatura no existe")

17. KeyError correcto: S99/temperatura no existe


In [38]:
procesador.current_value("S99", "temperatura")

KeyError: 

---

In [40]:
from Proyecto1.evaluador import *

In [41]:
import os
os.chdir('/workspaces/ED_2026_02_v2/Proyecto1')
print(os.getcwd())

/workspaces/ED_2026_02_v2/Proyecto1


In [42]:
ejecutar(DataProcessor, estudiante='Daniel Núñez')

EVALUACION - Data Stream Processor
Estudiante : Daniel Núñez
Fecha      : 2026-09-25 16:34:12
Python     : 3.12.13 (Linux)
Evaluador  : v1.1
Clase      : DataProcessor (modulo __main__)

== Interfaz y restricciones ==

[PASS ] Define add, process_next, undo, pending y current_value
[PASS ] Se puede crear DataProcessor() sin argumentos
[PASS ] __init__ crea una ArrayQueue, un ArrayStack y una list
[PASS ] No sustituye Queue/Stack por deque u otra estructura
[PASS ] ArrayQueue y ArrayStack son las del curso (no reimplementadas)

   Interfaz y restricciones: 5/5 OK

== Pruebas obligatorias (1-17) ==

[PASS ] 1. pending() sobre un procesador vacio
[PASS ] 2. Agregar un registro
[PASS ] 3. Agregar varios registros
[PASS ] 4. Procesamiento FIFO
[PASS ] 5. Procesar un registro
[PASS ] 6. Procesar varios registros
[PASS ] 7. Actualizar una variable existente
[PASS ] 8. Consultar el valor actual
[PASS ] 9. Realizar un undo()
[PASS ] 10. Varios undo() consecutivos
[PASS ] 11. process_next() con 